# Chat UI only (no training)
Loads the base model in 4-bit plus your saved LoRA adapter and opens a Gradio chat. Use after `train.ipynb`.
Runtime > T4 GPU. Set `ADAPTER` to your Hugging Face repo, or a Drive path such as `/content/drive/MyDrive/air-climate-lora` (mount Drive first).

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate gradio

In [ ]:
import torch, gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
BASE, ADAPTER = "Qwen/Qwen2.5-3B-Instruct", "lisaluis717/air-climate-lora"
SYSTEM = ("You are an air-quality and climate assistant for India. Explain ENSO, PM2.5 and pollution meteorology "
          "in plain language using the facts you were trained on. If you are not sure, say so and point to IMD, CPCB or the original study.")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto", dtype=torch.float16), ADAPTER).eval()

def respond(message, history):
    msgs = [{"role":"system","content":SYSTEM}] + [{"role":m["role"],"content":m["content"]} for m in history] + [{"role":"user","content":message}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad(): out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

gr.ChatInterface(respond, type="messages", title="Air Quality & Climate Assistant (India)",
    description="Fine-tuned Qwen (QLoRA, 4-bit). Informational only; verify with IMD / CPCB.",
    examples=["What is ENSO?", "Why did pollution move south in 2022-23?", "What is an airshed?"]).launch(share=True, debug=True)